# Bias Variance Trade-Off

In this notebook we are going to conduct a Bias-Variance Trade-off analysis. We are going to do this by implementing a ladder structure to explore our data to see what could have caused our model to not learn well on data that were bellow 500 to 999 reviews. 

Somewhat of the IDEA of LADDER:
1. Oregon, 3 grapes (Pinot Noir, Chardonnay, Pinot Gris) 
2. California, top 5 grapes
3. California, all grapes
4. All US
5. All Countries

In [ ]:
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

df = pd.read_pickle("model_data.pkl")

MODELS = {
    "M0 NB counts": make_pipeline(CountVectorizer(min_df=2, stop_words="english"), MultinomialNB()),
    "M1 LogReg unigrams": make_pipeline(TfidfVectorizer(min_df=2), LogisticRegression(max_iter=1000)),
    "M2 LogReg 1-3 grams": make_pipeline(TfidfVectorizer(ngram_range=(1, 3), min_df=2, sublinear_tf=True),
                                         LogisticRegression(max_iter=1000)),
}

LADDER = {
    "1 Oregon, 3 grapes": (df.province == "Oregon") & df.grape.isin(["Pinot Noir", "Chardonnay", "Pinot Gris"]),
    "2 California, top 5": (df.province == "California") & df.grape.isin(
                               df[df.province == "California"].grape.value_counts().index[:5]),
    "3 California, all": df.province == "California",
    "4 US": df.country == "US",
    "5 All countries": df.country.notna(),
}

def run_step(rung, model_name, min_per_class=30):
    part = df[LADDER[rung]]
    counts = part.grape.value_counts()
    part = part[part.grape.isin(counts[counts >= min_per_class].index)] # drop classes too small to learn
    tr, va = part[part.split == "train"], part[part.split == "val"]

    model = clone(MODELS[model_name]).fit(tr.description_masked, tr.grape)
    train_f1 = f1_score(tr.grape, model.predict(tr.description_masked), average="macro")
    val_f1   = f1_score(va.grape, model.predict(va.description_masked), average="macro")
    return {"rung": rung, "model": model_name, "n_train": len(tr), "n_classes": part.grape.nunique(),
            "train_f1": train_f1, "val_f1": val_f1, "gap": train_f1 - val_f1}
